# Notebook 2: Transactional Outbox + Polling Publisher

The fix for the dual-write problem is surprisingly boring: **use one transaction**. We store the outbound event as a row in an `outbox` table in the same DB, in the same transaction as the business data. A separate **publisher** reads the outbox and ships events to the bus.

- One commit -> both the order row *and* the event row exist, or neither does.
- The publisher is decoupled from the request handler. A slow or down bus no longer breaks order creation.
- The publisher is **at-least-once**: it may re-send after a crash, so consumers must be **idempotent** (we'll demo this below).

## Setup

```bash
cd 04-patterns/outbox-and-cdc
docker compose up -d
uv sync
```
Select the `.venv` kernel in VS Code.

## Schema + transactional write

Key design choices in the `outbox` table:

- `event_id UUID` - a stable, globally-unique ID *per event*. Consumers use it to deduplicate.
- `aggregate_id` - the business entity the event belongs to (e.g. `order:42`). Lets us route events for the same order to the same Kafka partition so they stay ordered.
- `topic` - logical routing key (`order.placed`, `order.shipped`, ...).
- `payload JSONB` - the event body.
- `created_at` / `published_at` - who wrote what when, and whether the publisher has shipped it yet.

In [1]:
import psycopg, json, uuid
DSN = 'host=localhost port=5432 user=demo password=demo dbname=outbox_demo'

with psycopg.connect(DSN, autocommit=True) as conn:
    conn.execute('DROP TABLE IF EXISTS orders')
    conn.execute('DROP TABLE IF EXISTS outbox')
    conn.execute('CREATE TABLE orders (id SERIAL PRIMARY KEY, item TEXT, total INTEGER)')
    conn.execute('''
        CREATE TABLE outbox (
            id             BIGSERIAL PRIMARY KEY,
            event_id       UUID NOT NULL UNIQUE,
            aggregate_id   TEXT NOT NULL,
            topic          TEXT NOT NULL,
            payload        JSONB NOT NULL,
            created_at     TIMESTAMPTZ NOT NULL DEFAULT now(),
            published_at   TIMESTAMPTZ
        )''')
    # Partial index: the publisher's hot query only scans unpublished rows
    conn.execute('CREATE INDEX outbox_unpublished_idx ON outbox(id) WHERE published_at IS NULL')

def place_order(item, total):
    with psycopg.connect(DSN) as conn:
        with conn.transaction():
            cur = conn.execute(
                'INSERT INTO orders(item,total) VALUES (%s,%s) RETURNING id',
                (item, total),
            )
            order_id = cur.fetchone()[0]
            conn.execute(
                '''INSERT INTO outbox(event_id, aggregate_id, topic, payload)
                   VALUES (%s, %s, %s, %s::jsonb)''',
                (
                    str(uuid.uuid4()),
                    f'order:{order_id}',
                    'order.placed',
                    json.dumps({'order_id': order_id, 'item': item, 'total': total}),
                ),
            )
        return order_id

for it in [('book', 25), ('pen', 5), ('lamp', 40)]:
    place_order(*it)

with psycopg.connect(DSN) as conn:
    print('orders:')
    for r in conn.execute('SELECT * FROM orders').fetchall(): print(' ', r)
    print('outbox:')
    for r in conn.execute('SELECT id, event_id, aggregate_id, topic, payload, published_at FROM outbox').fetchall():
        print(' ', r)

orders:
  (1, 'book', 25)
  (2, 'pen', 5)
  (3, 'lamp', 40)
outbox:
  (1, UUID('380b3b6f-61b7-4902-8ac7-629b737a7ce8'), 'order:1', 'order.placed', {'item': 'book', 'total': 25, 'order_id': 1}, None)
  (2, UUID('b9f4a60d-3a7f-45de-afcc-1b85420fe018'), 'order:2', 'order.placed', {'item': 'pen', 'total': 5, 'order_id': 2}, None)
  (3, UUID('f82a3cbf-249c-4ff1-976c-f3a280497eb8'), 'order:3', 'order.placed', {'item': 'lamp', 'total': 40, 'order_id': 3}, None)


## Rollback safety: no phantom events

If the business logic fails, the whole transaction rolls back - including the outbox row. This is exactly the guarantee we couldn't get with dual writes.

In [2]:
with psycopg.connect(DSN) as conn:
    before = conn.execute('SELECT count(*) FROM outbox').fetchone()[0]

try:
    with psycopg.connect(DSN) as conn:
        with conn.transaction():
            conn.execute('INSERT INTO orders(item,total) VALUES (%s,%s)', ('phantom', 999))
            conn.execute(
                '''INSERT INTO outbox(event_id, aggregate_id, topic, payload)
                   VALUES (%s, %s, %s, %s::jsonb)''',
                (str(uuid.uuid4()), 'order:phantom', 'order.placed', json.dumps({'item': 'phantom'})),
            )
            raise RuntimeError('business rule failed')
except RuntimeError as e:
    print(e)

with psycopg.connect(DSN) as conn:
    after = conn.execute('SELECT count(*) FROM outbox').fetchone()[0]
    orders = conn.execute("SELECT count(*) FROM orders WHERE item='phantom'").fetchone()[0]
print(f'outbox rows before={before}  after={after}   phantom orders={orders}')
print('>> the failed transaction rolled back BOTH the order and the outbox row')

business rule failed
outbox rows before=3  after=3   phantom orders=0
>> the failed transaction rolled back BOTH the order and the outbox row


## Polling publisher

A small loop picks up unpublished rows, ships them to the bus, and marks them published. Two important tricks:

- `FOR UPDATE SKIP LOCKED` - multiple publisher workers can run in parallel without blocking each other or double-publishing the same row.
- `ORDER BY id` - preserves insertion order globally. If you care about per-entity order, partition downstream by `aggregate_id` (notebook 4 shows how).

In [3]:
bus = []  # pretend Kafka / RabbitMQ

def publish_round(batch_size=10):
    with psycopg.connect(DSN) as conn:
        with conn.transaction():
            rows = conn.execute('''
                SELECT id, event_id, aggregate_id, topic, payload
                FROM outbox
                WHERE published_at IS NULL
                ORDER BY id
                LIMIT %s
                FOR UPDATE SKIP LOCKED
            ''', (batch_size,)).fetchall()
            for row_id, event_id, aggregate_id, topic, payload in rows:
                bus.append({
                    'event_id': str(event_id),
                    'key': aggregate_id,   # partition key - keeps per-entity order
                    'topic': topic,
                    'payload': payload,
                })
                conn.execute('UPDATE outbox SET published_at = now() WHERE id = %s', (row_id,))
        return len(rows)

n = publish_round()
print(f'published {n} events')
for e in bus: print(' ', e)

published 3 events
  {'event_id': '380b3b6f-61b7-4902-8ac7-629b737a7ce8', 'key': 'order:1', 'topic': 'order.placed', 'payload': {'item': 'book', 'total': 25, 'order_id': 1}}
  {'event_id': 'b9f4a60d-3a7f-45de-afcc-1b85420fe018', 'key': 'order:2', 'topic': 'order.placed', 'payload': {'item': 'pen', 'total': 5, 'order_id': 2}}
  {'event_id': 'f82a3cbf-249c-4ff1-976c-f3a280497eb8', 'key': 'order:3', 'topic': 'order.placed', 'payload': {'item': 'lamp', 'total': 40, 'order_id': 3}}


## At-least-once: what if the publisher crashes mid-round?

Imagine the publisher sends the event to the bus, but the process dies *before* the `UPDATE outbox SET published_at = now()` commits. On restart, it will re-send the same row. Consumers can see the same event more than once - the hallmark of *at-least-once* delivery.

The fix is consumer-side **idempotency**, keyed by `event_id`:

In [4]:
# Simulate a consumer that applies each event exactly once by remembering event_id
applied = {}
seen = set()

def consume(event):
    if event['event_id'] in seen:
        return 'duplicate-ignored'
    seen.add(event['event_id'])
    applied[event['key']] = event['payload']
    return 'applied'

# First delivery
results1 = [consume(e) for e in bus]
# Redelivery of the exact same events (simulating publisher retry after crash)
results2 = [consume(e) for e in bus]

print('first  delivery:', results1)
print('second delivery:', results2)
print('final state    :', applied)
print('>> duplicates rejected by event_id - consumer is idempotent')

first  delivery: ['applied', 'applied', 'applied']
second delivery: ['duplicate-ignored', 'duplicate-ignored', 'duplicate-ignored']
final state    : {'order:1': {'item': 'book', 'total': 25, 'order_id': 1}, 'order:2': {'item': 'pen', 'total': 5, 'order_id': 2}, 'order:3': {'item': 'lamp', 'total': 40, 'order_id': 3}}
>> duplicates rejected by event_id - consumer is idempotent


## Retention: don't let the outbox grow forever

In production the outbox is hot-path write traffic. If you never delete, it turns into your largest table. Two common strategies:

1. **Delete on publish** - the publisher `DELETE`s the row instead of setting `published_at`. Simple, but you lose audit trail.
2. **Archive** - keep `published_at IS NOT NULL` rows for N days, then `DELETE` them on a cron. Keeps a short audit window cheaply.

Either way, `VACUUM` matters. The partial index `WHERE published_at IS NULL` keeps the hot query fast even as the table grows.

In [5]:
with psycopg.connect(DSN, autocommit=True) as conn:
    deleted = conn.execute("DELETE FROM outbox WHERE published_at < now() - interval '7 days'").rowcount
print(f'pruned {deleted} old outbox rows (none yet - they were all just written)')

pruned 0 old outbox rows (none yet - they were all just written)


## Summary

| Concern | How outbox handles it |
|---|---|
| Atomicity (DB + event) | Same transaction, same DB - one commit |
| Duplicates after crash | At-least-once; consumer dedups by `event_id` |
| Ordering | Global: `ORDER BY id`. Per-entity: partition by `aggregate_id` |
| Parallel publishers | `FOR UPDATE SKIP LOCKED` |
| Table bloat | Prune or delete-on-publish + partial index |

The main cost is **polling latency** (you wait for the next poll tick). Notebook 3 removes polling entirely by reading the database's write-ahead log directly.